# `Part 03` — Modelling poem trajectories

*Which poems come into fashion, and which fall out?* This is where we narrow from the corpus to the individual poem. We build a poem × decade grid — each cell holding `quoted_works` (distinct host works that quote the poem in that decade) over `n_works` (host works in scope) — and fit a **Bayesian hierarchical beta-binomial GLMM** to the breadth question: *for a host work in this decade, what's the probability it quotes this poem?*

Two decisions shape everything downstream and are handled here: **left-truncation** (a poem only enters its own at-risk window once its poet could plausibly have written it) and a **length-bias** check. After the fit we run the validation suite — a quasi-binomial companion, the shrinkage diagnostic, variance partition, and a posterior predictive check — and then persist the fitted model so `04_findings` can read it back without re-sampling.

This notebook reads `excerpts_df` and the exposure / host-metadata tables from notebook 01. The NUTS fit in §2 takes ~5–10 minutes; everything after it is cheap.

In [1]:
# --- Setup: load the corpus built in notebook 01 ----------------------------
import sys, time as _time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import altair as alt
import statsmodels.api as sm
import patsy

for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "paths.py").is_file():
        sys.path.insert(0, str(_p / "src"))
        break
else:
    raise FileNotFoundError(
        "Could not find src/paths.py — run this notebook with cwd = repository root"
    )

from paths import REPO_ROOT, DATA_DIR, EXPORTS_DIR, MODEL_DIR, CACHE_DIR, passim_data_paths

data_paths = passim_data_paths()
alt.data_transformers.disable_max_rows()

# frames persisted by notebook 01
excerpts_df                = pl.read_parquet(EXPORTS_DIR / "excerpts_df.parquet")
exposure_year_df           = pl.read_parquet(EXPORTS_DIR / "exposure_year_df.parquet")
exposure_decade_df         = pl.read_parquet(EXPORTS_DIR / "exposure_decade_df.parquet")
host_universe_work_meta_df = pl.read_parquet(EXPORTS_DIR / "host_universe_work_meta_df.parquet")
work_meta_df               = pl.read_parquet(EXPORTS_DIR / "work_meta_df.parquet")

# knobs — must match notebook 01 / the source config
EXPO_DENOM = "total_pages"
POET_AGE_AT_RISK = 18
POET_DEATH_LOOKBACK = 70
_PLOT_FLOOR = 1e-6
PLOT_Y_SCALE = "linear"
PREVALENCE_DEVIATION_SCALE = "log_odds"
INTENSITY_DEVIATION_SCALE = "log_rate_ratio"

# model window: first year with >= N host works (same logic as the source config)
MIN_WORKS_PER_YEAR_FOR_AUTO_WINDOW = 5
MODEL_YEAR_MIN_OVERRIDE = None
_auto = exposure_year_df.filter(pl.col("n_works") >= MIN_WORKS_PER_YEAR_FOR_AUTO_WINDOW)
MODEL_YEAR_MIN_AUTO = int(_auto["ppa_pub_year"].min()) if _auto.height > 0 else int(exposure_year_df["ppa_pub_year"].min())
MODEL_YEAR_MIN = int(MODEL_YEAR_MIN_OVERRIDE) if MODEL_YEAR_MIN_OVERRIDE is not None else MODEL_YEAR_MIN_AUTO

# `_dated_slice_5`: dated rows with a non-empty poem_id (used by the grid builder)
_y = pl.col("ppa_pub_year").cast(pl.Int32, strict=False)
_pid = pl.col("poem_id").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars() != ""
_dated_slice_5 = excerpts_df.filter(_pid & _y.is_not_null())

print(f"excerpts_df: {excerpts_df.shape} | MODEL_YEAR_MIN: {MODEL_YEAR_MIN}")

excerpts_df: (616670, 42) | MODEL_YEAR_MIN: 1694


## 1. Building the poem × decade grid

*Which poems come into fashion and which fall out?* Raw excerpt totals are not a clean starting point for that question. They mix two different mechanisms. A poem can spread to **more host works** (breadth), or a poem can already be widespread but be treated **more heavily** inside the works that quote it (depth). This section builds the data structure our primary model will sit on top of — a poem × period grid in which every cell holds a numerator `quoted_works` (distinct host works that quote this poem in this period) and a denominator `n_works` (distinct host works in scope for this period). From that grid we can ask the breadth question cleanly: *for a host work in this period, what is the probability it quotes this poem?*

Because yearly data are extremely sparse for most poems (many poems get one or two excerpts per decade at best), the default period grain is the **decade**. That gives us a cleaner baseline and lets missing periods count as real zeros rather than silently disappearing from the design matrix.

Two decisions shape every downstream statistic. This subsection explains the first (left-truncation); §1.3 explains the second (length-bias diagnostic).

### 1.1 Left-truncation — when does a poem enter its own at-risk window?

Longfellow's *Christus* (1872) cannot possibly be in the prosodic discourse of 1710: Longfellow was born in 1807. A 1710 cell in Longfellow's row of the prevalence grid is either a spurious Passim alignment or a structural zero — the poem *could not exist* there. Naively counting those cells as real zero observations would make Longfellow look like a dramatic riser simply because he could not have been quoted earlier. We therefore **left-truncate** the poem × period grid: for each poem we drop all periods before the poem's plausible at-risk window begins.

The priority chain is identical to the temporal filter in notebook 01, so the two stages tell a consistent story. For each poem we anchor the at-risk window using the *first* of the following that is available:

1. **`wd_birth + POET_AGE_AT_RISK`** — Wikidata birth year plus the adult-life offset. This is the primary anchor: the earliest year the poet could plausibly compose the poem.
2. **`ch_birth_lo + POET_AGE_AT_RISK`** — Chadwyck lower-bound birth fallback, used when Wikidata birth is missing.
3. **`wd_death − POET_DEATH_LOOKBACK`** — death-only fallback when no birth anchor exists. Assumes the poet was active for the last `POET_DEATH_LOOKBACK` years before death.
4. **`first_ppa_year − 10`** — empirical fallback for anonymous works with no author dates at all.
5. **`MODEL_YEAR_MIN`** — ultimate fallback for ancient and universally anonymous works (Bible, Homer).

`RECEPTION_LAG_DECADES` (default `0`) can delay the at-risk start by a further number of decades past the anchor — useful if you want to allow for a typical lag between composition and broader circulation.

This makes the slope for each poem mean the same thing: **fashion change within the poem's own at-risk window**, not a mixture of fashion change and a structural ramp-up from non-existence.

### Knobs in the code cell below

- **`TOP_N_POEMS`** *(24)* — how many support-qualified poems to highlight in the downstream small-multiples plots (notebook 04). Changes the display, not the modelled population.
- **`POEM_PERIOD`** *(`"decade"`)* — time grain for the prevalence and intensity grids.
- **`PREV_SUPPORT_MIN_LIFETIME_WORKS`** *(25)* — minimum distinct host works across the full window for a poem to enter the modelled population. Filters out long-tail one-off detections.
- **`PREV_SUPPORT_MIN_PERIODS`** *(10)* — minimum observed periods for a poem to enter the modelled population. Rejects poems with too few anchor points to trace a trajectory.
- **`PREV_SPLINE_DF_GRID`** *(3–6)* — AIC search grid for the spline df on the shared trend term, used by the quasi-binomial companion in §3.
- **`POEM_DEVIATION_TYPE`** *(`"cubic_spline"`)* — basis used for each poem's *deviation* from the population trend in the quasi-binomial companion: `"cubic_spline"`, `"quadratic"`, or `"linear"`.
- **`POEM_DEVIATION_DF`** *(3)* — degrees of freedom for the cubic-spline deviation basis (ignored when `POEM_DEVIATION_TYPE ≠ "cubic_spline"`).
- **`APPLY_LEFT_TRUNCATION`** *(`True`)* — toggle for the at-risk truncation described above. Disable for sensitivity checks against the full grid.
- **`RECEPTION_LAG_DECADES`** *(0)* — additional delay (in decades) added to the anchor year before the poem becomes at-risk.

In [2]:
TOP_N_POEMS = 12                       # how many poems to highlight in small-multiples plots
POEM_PERIOD = "decade"                 # "decade" (recommended) | "year" for small experiments

PREV_SUPPORT_MIN_LIFETIME_WORKS = 30    # min distinct host works, lifetime, for a poem to enter the model, default 25
PREV_SUPPORT_MIN_PERIODS = 10            # min observed periods for a poem to enter the model, default 10

INTENSITY_METRIC = "pages"             # "pages" | "excerpts" — response for the conditional-intensity model
PREV_SPLINE_DF_GRID = list(range(5, 6)) # default 4-7 - but we know 5 is best
INTENSITY_SPLINE_DF_GRID = list(range(5, 6)) # default 4-7 - but we know 5 is best

POEM_DEVIATION_TYPE = "cubic_spline"   # "cubic_spline" | "quadratic" | "linear" # default cubic_spline
POEM_DEVIATION_DF = 3                  # only used when POEM_DEVIATION_TYPE == "cubic_spline"

# Left-truncation controls (uses POET_AGE_AT_RISK / POET_DEATH_LOOKBACK from the setup cell — same cascade as notebook 01's temporal filter)
APPLY_LEFT_TRUNCATION = True
RECEPTION_LAG_DECADES = 0              # extra delay (decades) beyond the at-risk anchor

assert POEM_PERIOD in ("year", "decade")
assert INTENSITY_METRIC in ("pages", "excerpts")
assert POEM_DEVIATION_TYPE in ("cubic_spline", "quadratic", "linear")


def _safe_prob(values, eps: float = 1e-6):
    _arr = np.asarray(values, dtype=float)
    return np.clip(_arr, eps, 1 - eps)


def _safe_logit(values, eps: float = 1e-6):
    _p = _safe_prob(values, eps=eps)
    return np.log(_p / (1 - _p))


def _prevalence_dev_values(fit_values, baseline_values, scale: str):
    _fit = np.asarray(fit_values, dtype=float)
    _base = np.asarray(baseline_values, dtype=float)
    if scale == "log_odds":
        return (
            _safe_logit(_fit) - _safe_logit(_base),
            "log-odds difference from baseline",
            0.0,
        )
    if scale == "odds_ratio":
        return (
            np.exp(_safe_logit(_fit) - _safe_logit(_base)),
            "odds ratio vs baseline",
            1.0,
        )
    return (
        _fit / np.clip(_base, 1e-6, None),
        "relative risk vs baseline",
        1.0,
    )


def _intensity_dev_values(fit_values, baseline_values, scale: str):
    _fit = np.asarray(fit_values, dtype=float)
    _base = np.asarray(baseline_values, dtype=float)
    if scale == "log_rate_ratio":
        return (
            np.log(np.clip(_fit, 1e-6, None) / np.clip(_base, 1e-6, None)),
            "log rate ratio vs baseline",
            0.0,
        )
    return (
        _fit / np.clip(_base, 1e-6, None),
        "rate ratio vs baseline",
        1.0,
    )


def _deviation_domain(values, ref_value: float):
    _s = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if _s.empty:
        return [-1.0, 1.0] if ref_value == 0 else [0.0, 2.0]
    if ref_value == 0:
        _lim = float(_s.abs().quantile(0.98)) * 1.15
        _lim = max(_lim, 0.25)
        return [-_lim, _lim]
    _hi = float(_s.quantile(0.99)) * 1.10
    _hi = max(_hi, 1.25)
    return [0.0, _hi]


def _poem_label_expr() -> pl.Expr:
    _title = pl.col("poem_title").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars()
    _pid = pl.col("poem_id").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars()
    return pl.when(_title != "").then(_title).otherwise(_pid).alias("poem_label")


_period_expr = (
    pl.col("ppa_pub_year").cast(pl.Int32, strict=False).alias("period")
    if POEM_PERIOD == "year"
    else (pl.col("ppa_pub_year").cast(pl.Int32, strict=False).floordiv(10) * 10)
    .cast(pl.Int32)
    .alias("period")
)
_POEM_PERIOD_MIN = MODEL_YEAR_MIN if POEM_PERIOD == "year" else (MODEL_YEAR_MIN // 10) * 10
_expo_period_pd = (
    exposure_year_df.to_pandas().rename(columns={"ppa_pub_year": "period"})
    if POEM_PERIOD == "year"
    else exposure_decade_df.to_pandas().rename(columns={"ppa_pub_decade": "period"})
)
_expo_period_pd = _expo_period_pd[_expo_period_pd["period"] >= _POEM_PERIOD_MIN].copy()

_poem_base_pl = (
    _dated_slice_5
    .with_columns(
        pl.col("poem_id").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars().alias("poem_id"),
        pl.col("ppa_pub_year").cast(pl.Int32, strict=False).alias("year"),
        _period_expr,
        _poem_label_expr(),
    )
    .filter(
        (pl.col("poem_id") != "")
        & pl.col("year").is_not_null()
        & (pl.col("year") >= MODEL_YEAR_MIN)
        & (pl.col("period") >= _POEM_PERIOD_MIN)
    )
)

_poem_support_pl = (
    _poem_base_pl
    .group_by("poem_id")
    .agg(
        pl.col("poem_label").drop_nulls().first().alias("poem_label"),
        pl.col("ppa_work_id").n_unique().alias("lifetime_works"),
        pl.col("page_id").n_unique().alias("lifetime_pages"),
        pl.col("period").n_unique().alias("n_periods_observed"),
    )
    .sort(["lifetime_works", "n_periods_observed"], descending=True)
)

_modeled_poems_pl = _poem_support_pl.filter(
    (pl.col("lifetime_works") >= PREV_SUPPORT_MIN_LIFETIME_WORKS)
    & (pl.col("n_periods_observed") >= PREV_SUPPORT_MIN_PERIODS)
)
_plot_poems_pd = _modeled_poems_pl.head(TOP_N_POEMS).to_pandas()
_plot_poem_ids = set(_plot_poems_pd["poem_id"])

_prev_obs_pl = (
    _poem_base_pl
    .join(_modeled_poems_pl.select("poem_id"), on="poem_id", how="inner")
    .group_by(["poem_id", "period"])
    .agg(
        pl.col("poem_label").drop_nulls().first().alias("poem_label"),
        pl.col("ppa_work_id").n_unique().alias("quoted_works"),
        pl.col("page_id").n_unique().alias("quoted_pages"),
        pl.len().alias("quoted_excerpts"),
    )
    .sort(["poem_id", "period"])
)
_prev_obs_pd = _prev_obs_pl.to_pandas()

_poem_meta_pd = _modeled_poems_pl.to_pandas().sort_values("poem_id").reset_index(drop=True)
_periods_pd = pd.DataFrame({"period": sorted(_expo_period_pd["period"].unique())})
_prev_df = _poem_meta_pd.assign(_k=1).merge(_periods_pd.assign(_k=1), on="_k").drop(columns="_k")
_prev_df = (
    _prev_df
    .merge(
        _prev_obs_pd[["poem_id", "period", "quoted_works", "quoted_pages", "quoted_excerpts"]],
        on=["poem_id", "period"],
        how="left",
    )
    .merge(_expo_period_pd[["period", "n_works", "total_pages"]], on="period", how="left")
)
for _col in ["quoted_works", "quoted_pages", "quoted_excerpts"]:
    _prev_df[_col] = _prev_df[_col].fillna(0).astype(int)

_prev_df = _prev_df[_prev_df["n_works"] > 0].copy()

_author_dates_pl = (
    _dated_slice_5
    .select(
        "poem_id",
        pl.col("ref_md_birth_year_wd").cast(pl.Int32, strict=False).alias("birth_wd"),
        pl.col("ref_md_death_year_wd").cast(pl.Int32, strict=False).alias("death_wd"),
        pl.col("ref_md_ch_birth_lo").cast(pl.Int32, strict=False).alias("ch_birth_lo"),
    )
    .group_by("poem_id")
    .agg(
        pl.col("birth_wd").drop_nulls().first().alias("birth_wd"),
        pl.col("death_wd").drop_nulls().first().alias("death_wd"),
        pl.col("ch_birth_lo").drop_nulls().first().alias("ch_birth_lo"),
    )
)
_first_ppa_pl = (
    _dated_slice_5
    .with_columns(pl.col("ppa_pub_year").cast(pl.Int32, strict=False).alias("yr"))
    .filter(pl.col("yr").is_not_null())
    .group_by("poem_id")
    .agg(pl.col("yr").min().alias("first_ppa_year"))
)
_author_dates_pd = _author_dates_pl.join(_first_ppa_pl, on="poem_id", how="left").to_pandas()

def _compute_at_risk_year(row):
    """Earliest at-risk year for a poem, matching notebook 01's temporal filter cascade.

    Priority (first available wins), + `RECEPTION_LAG_DECADES * 10` on top:
      1. WD birth + POET_AGE_AT_RISK            (primary; author adult life)
      2. CH lower-bound birth + POET_AGE_AT_RISK (Chadwyck fallback)
      3. WD death - POET_DEATH_LOOKBACK          (death-only fallback)
      4. first PPA appearance - 10               (empirical fallback, anonymous works)
      5. MODEL_YEAR_MIN                          (ultimate fallback)
    """
    # Accept BCE / negative years — they are valid Wikidata birth/death values
    # for classical authors (Homer, Virgil, Horace, …). The .clip(lower=
    # MODEL_YEAR_MIN) downstream keeps pre-window dates from actually
    # truncating those poems; they just anchor as wd_birth+POET_AGE_AT_RISK
    # and effectively get the full modelled window.
    _lag = RECEPTION_LAG_DECADES * 10
    if pd.notna(row["birth_wd"]):
        return int(row["birth_wd"]) + POET_AGE_AT_RISK + _lag, f"wd_birth+{POET_AGE_AT_RISK}"
    if pd.notna(row["ch_birth_lo"]):
        return int(row["ch_birth_lo"]) + POET_AGE_AT_RISK + _lag, f"ch_birth+{POET_AGE_AT_RISK}"
    if pd.notna(row["death_wd"]):
        return int(row["death_wd"]) - POET_DEATH_LOOKBACK + _lag, f"wd_death-{POET_DEATH_LOOKBACK}"
    if pd.notna(row["first_ppa_year"]):
        return int(row["first_ppa_year"]) - 10, "first_ppa-10"
    return MODEL_YEAR_MIN, "model_min"

_author_dates_pd[["at_risk_year_raw", "at_risk_source"]] = (
    _author_dates_pd.apply(_compute_at_risk_year, axis=1, result_type="expand")
)
_author_dates_pd["at_risk_year"] = _author_dates_pd["at_risk_year_raw"].clip(lower=MODEL_YEAR_MIN)
if POEM_PERIOD == "decade":
    _author_dates_pd["at_risk_period"] = (_author_dates_pd["at_risk_year"] // 10 * 10).astype(int)
else:
    _author_dates_pd["at_risk_period"] = _author_dates_pd["at_risk_year"].astype(int)

print("\nLeft-truncation: source of 'at-risk start year' per poem")
_branch_counts = (
    _author_dates_pd.merge(_modeled_poems_pl.select("poem_id").to_pandas(), on="poem_id", how="inner")
    ["at_risk_source"].value_counts()
)
for _src, _n in _branch_counts.items():
    print(f"  {_src:20s}: {_n:4d} poems")

# Surface the `first_ppa-10` cohort: these are poems with no author dates at all
# (no WD birth / death, no CH birth-lo). Worth eyeballing to see what actually lands
# in the circular "use the first PPA year as its own anchor" fallback.
_firstppa_cohort = (
    _author_dates_pd
    .merge(_modeled_poems_pl.select(["poem_id", "poem_label", "lifetime_works", "n_periods_observed"]).to_pandas(),
           on="poem_id", how="inner")
    .loc[lambda d: d["at_risk_source"] == "first_ppa-10"]
    .sort_values("lifetime_works", ascending=False)
    .loc[:, ["poem_id", "poem_label", "first_ppa_year", "at_risk_year",
             "lifetime_works", "n_periods_observed"]]
    .reset_index(drop=True)
)
if len(_firstppa_cohort) > 0:
    print(
        f"\n`first_ppa-10` cohort ({len(_firstppa_cohort):,} poems — no author date anchor available):"
    )
    pd.set_option("display.max_colwidth", 55)
    print(_firstppa_cohort.head(20).to_string(index=False))
    if len(_firstppa_cohort) > 20:
        print(f"  … {len(_firstppa_cohort) - 20} more. Full table: _firstppa_cohort")

_prev_df = _prev_df.merge(
    _author_dates_pd[["poem_id", "at_risk_year", "at_risk_period", "at_risk_source",
                      "birth_wd", "death_wd", "first_ppa_year"]],
    on="poem_id",
    how="left",
)
_prev_df["at_risk_year"] = _prev_df["at_risk_year"].fillna(MODEL_YEAR_MIN).astype(int)
_prev_df["at_risk_period"] = _prev_df["at_risk_period"].fillna(_POEM_PERIOD_MIN).astype(int)

_n_before_trunc = len(_prev_df)
_n_pre_risk = int((_prev_df["period"] < _prev_df["at_risk_period"]).sum())
_n_pre_risk_with_quotes = int(
    ((_prev_df["period"] < _prev_df["at_risk_period"]) & (_prev_df["quoted_works"] > 0)).sum()
)

print(f"\nPrevalence grid rows BEFORE truncation : {_n_before_trunc:,}")
print(f"  cells with period < at_risk_period   : {_n_pre_risk:,}")
print(f"  of which have quoted_works > 0       : {_n_pre_risk_with_quotes:,}")
print(f"    → these are 'spurious pre-existence' PPA detections;")
print(f"      the left-truncation removes them from the prevalence model.")

_pre_birth_spurious = _prev_df[
    (_prev_df["birth_wd"].notna())
    & (_prev_df["period"] < _prev_df["birth_wd"])
    & (_prev_df["quoted_works"] > 0)
].copy()
if len(_pre_birth_spurious) > 0:
    print(f"\n  Sample spurious pre-BIRTH Passim detections (by author Wikidata birth):")
    _top_spur = _pre_birth_spurious.sort_values("quoted_works", ascending=False).head(6)
    for _, r in _top_spur.iterrows():
        print(
            f"    {str(r['poem_label'])[:45]:45s}  born={int(r['birth_wd'])}  "
            f"period={int(r['period'])}  quoted_works={int(r['quoted_works'])}"
        )

if APPLY_LEFT_TRUNCATION:
    _prev_df_full = _prev_df.copy()
    _prev_df = _prev_df[_prev_df["period"] >= _prev_df["at_risk_period"]].copy()
    print(f"\nAfter left-truncation: {len(_prev_df):,} rows "
          f"({_n_before_trunc - len(_prev_df):,} dropped)")
else:
    print(f"\nLeft-truncation DISABLED (APPLY_LEFT_TRUNCATION = False)")

# Boundary-decade correction: if a poem becomes at-risk inside a decade,
# only hosts published on/after its at-risk year belong in that first denominator.
# Later decades keep the shared decade denominator.
_n_boundary_adjusted = 0
_n_boundary_zero_denom = 0
if POEM_PERIOD == "decade" and len(_prev_df) > 0:
    _host_period_pd = (
        host_universe_work_meta_df
        .select(
            pl.col("work_id").alias("ppa_work_id"),
            pl.col("pub_year").cast(pl.Int32, strict=False).alias("pub_year"),
            pl.col("page_count").fill_null(0).cast(pl.Int64).alias("page_count"),
        )
        .filter(pl.col("pub_year").is_not_null() & (pl.col("pub_year") >= MODEL_YEAR_MIN))
        .with_columns(period=(pl.col("pub_year").floordiv(10) * 10).cast(pl.Int32))
        .to_pandas()
    )
    _hosts_by_period = {
        int(_p): _g[["pub_year", "page_count"]].reset_index(drop=True)
        for _p, _g in _host_period_pd.groupby("period")
    }

    _boundary_mask = (
        (_prev_df["period"] == _prev_df["at_risk_period"])
        & (_prev_df["at_risk_year"] > _prev_df["period"])
    )
    _boundary_idx = _prev_df.index[_boundary_mask].to_list()
    _n_boundary_adjusted = len(_boundary_idx)

    for _idx in _boundary_idx:
        _row = _prev_df.loc[_idx]
        _hosts_here = _hosts_by_period.get(int(_row["period"]))
        if _hosts_here is None:
            _eligible_hosts = _hosts_here
        else:
            _eligible_hosts = _hosts_here[_hosts_here["pub_year"] >= int(_row["at_risk_year"])]
        _n_eligible = 0 if _eligible_hosts is None else int(len(_eligible_hosts))
        _pages_eligible = 0 if _eligible_hosts is None else int(_eligible_hosts["page_count"].sum())
        _prev_df.at[_idx, "n_works"] = _n_eligible
        _prev_df.at[_idx, "total_pages"] = _pages_eligible

    _n_boundary_zero_denom = int((_prev_df["n_works"] <= 0).sum())
    if _n_boundary_zero_denom:
        _prev_df = _prev_df[_prev_df["n_works"] > 0].copy()

    print(
        f"Boundary-decade denominator correction: {_n_boundary_adjusted:,} poem-decade cells adjusted; "
        f"{_n_boundary_zero_denom:,} cells dropped with no eligible host works."
    )

_prev_df["n_works"] = _prev_df["n_works"].astype(int)
_prev_df["total_pages"] = _prev_df["total_pages"].fillna(0).astype(int)
_prev_df["prop_quoted_raw"] = _prev_df["quoted_works"] / _prev_df["n_works"]
_prev_df["period_c"] = _prev_df["period"].astype(float) - _prev_df["period"].mean()
_prev_df["period_z"] = _prev_df["period_c"] / (10.0 if POEM_PERIOD == "decade" else 1.0)
_prev_df["is_plot_poem"] = _prev_df["poem_id"].isin(_plot_poem_ids)

_post_trunc_periods = _prev_df.groupby("poem_id").size()
_n_sparse_after = int((_post_trunc_periods < 5).sum())
if _n_sparse_after > 0:
    print(f"\n  Note: {_n_sparse_after} poems now have <5 periods after truncation; "
          f"trajectory estimates for these will be noisy.")

_intensity_count_col = "quoted_pages" if INTENSITY_METRIC == "pages" else "quoted_excerpts"
_intensity_label = (
    "quoted pages per quoting work"
    if INTENSITY_METRIC == "pages"
    else "excerpt rows per quoting work"
)
_int_df = _prev_df[_prev_df["quoted_works"] > 0].copy()
_int_df["count_metric"] = _int_df[_intensity_count_col].astype(float)
_int_df["log_off"] = np.log(_int_df["quoted_works"].clip(lower=1).astype(float))
_int_df["rate_raw"] = _int_df["count_metric"] / _int_df["quoted_works"].astype(float)

print("Poem-fashion prep:")
print(f"  period grain                 : {POEM_PERIOD}")
print(f"  modeled start                : {MODEL_YEAR_MIN}  → period floor {_POEM_PERIOD_MIN}")
print(
    f"  support threshold            : ≥{PREV_SUPPORT_MIN_LIFETIME_WORKS} lifetime works "
    f"and ≥{PREV_SUPPORT_MIN_PERIODS} observed periods"
)
print(f"  support-qualified poems      : {_modeled_poems_pl.height:,}")
print(f"  prevalence grid rows         : {len(_prev_df):,}")
print(f"  zero share in prevalence grid: {(_prev_df['quoted_works'] == 0).mean():.3f}")
print(f"  intensity rows (quoted only) : {len(_int_df):,}")
print(f"  intensity metric             : {_intensity_label}")
print("\nTop plot poems by lifetime quoted works:")
print(
    _plot_poems_pd[["poem_label", "lifetime_works", "n_periods_observed"]]
    .head(12)
    .to_string(index=False)
)

# save to csv
_prev_df.to_csv("prev_df.csv", index=False)



Left-truncation: source of 'at-risk start year' per poem
  wd_birth+18         :  874 poems
  first_ppa-10        :   27 poems
  ch_birth+18         :    5 poems

`first_ppa-10` cohort (27 poems — no author date anchor available):
   poem_id                                                          poem_label  first_ppa_year  at_risk_year  lifetime_works  n_periods_observed
Z200457892                                                        ODE on LIGHT            1676          1694             163                  23
Z200439528                                               OVID's METAMORPHOSES.            1697          1694             160                  22
Z200509644                                               Peace and Retirement.            1697          1694             151                  24
Z200206860                                     [Vital spark of heav'nly flame]            1752          1742             146                  17
Z300349916                                 

In [3]:
_exc_modelled = _poem_base_pl.join(
    _modeled_poems_pl.select("poem_id"), on="poem_id", how="inner"
)
_n_app = (
    0
    if _exc_modelled.height == 0
    else int(_exc_modelled.select(["poem_id", "ppa_work_id"]).unique().height)
)
_n_poems = _exc_modelled["poem_id"].n_unique()
_n_hosts = _exc_modelled["ppa_work_id"].n_unique()
print(
    f"  {_n_app:,} distinct appearances of {_n_poems:,} poems "
    f"across {_n_hosts:,} PPA host works "
    f"(model window + support-qualified poems)"
)

  93,358 distinct appearances of 906 poems across 3,619 PPA host works (model window + support-qualified poems)


### 1.2 Modelled population — what the trajectory model actually covers

The Bayesian trajectory model is intentionally not fit to the entire long tail of one-off Passim detections. It targets poems with sustained PPA circulation: enough distinct host works and enough observed periods to make a trajectory interpretable. The small report below makes that scope explicit before we move into modelling.

In [4]:
# Report the support-qualified population used by the Bayesian trajectory model.
# This keeps the article claim precise: we model sustained circulation, not the full long tail.
_modelled_ids = set(_modeled_poems_pl["poem_id"].to_list())
_base_for_scope = _poem_base_pl.to_pandas()
_modelled_scope = _base_for_scope[_base_for_scope["poem_id"].isin(_modelled_ids)].copy()

_all_poem_host_pairs = (
    _base_for_scope[["poem_id", "ppa_work_id"]]
    .drop_duplicates()
)
_modelled_poem_host_pairs = (
    _modelled_scope[["poem_id", "ppa_work_id"]]
    .drop_duplicates()
)

_scope_summary = pd.DataFrame({
    "quantity": [
        "post-filter poem ids in model window",
        "support-qualified poem ids modelled",
        "share of poem ids modelled",
        "post-filter excerpt rows in model window",
        "excerpt rows represented by modelled poems",
        "share of excerpt rows represented",
        "poem-host pairs in model window",
        "poem-host pairs represented by modelled poems",
        "share of poem-host pairs represented",
        "host works touched by modelled poems",
    ],
    "value": [
        f"{_base_for_scope['poem_id'].nunique():,}",
        f"{len(_modelled_ids):,}",
        f"{len(_modelled_ids) / _base_for_scope['poem_id'].nunique():.1%}",
        f"{len(_base_for_scope):,}",
        f"{len(_modelled_scope):,}",
        f"{len(_modelled_scope) / len(_base_for_scope):.1%}",
        f"{len(_all_poem_host_pairs):,}",
        f"{len(_modelled_poem_host_pairs):,}",
        f"{len(_modelled_poem_host_pairs) / len(_all_poem_host_pairs):.1%}",
        f"{_modelled_scope['ppa_work_id'].nunique():,}",
    ],
})

print(
    "Modelled population: sustained-circulation poems "
    f"(≥{PREV_SUPPORT_MIN_LIFETIME_WORKS} lifetime PPA works and "
    f"≥{PREV_SUPPORT_MIN_PERIODS} observed {POEM_PERIOD}s)."
)
print(
    "This model describes the support-qualified reception stratum, not every poem id "
    "that appears once in Passim."
)
_scope_summary

Modelled population: sustained-circulation poems (≥30 lifetime PPA works and ≥10 observed decades).
This model describes the support-qualified reception stratum, not every poem id that appears once in Passim.


,quantity,value
0,post-filter poem ids in model window,"37,403"
1,support-qualified poem ids modelled,906
2,share of poem ids modelled,2.4%
3,post-filter excerpt rows in model window,"612,066"
4,excerpt rows represented by modelled poems,"317,201"
5,share of excerpt rows represented,51.8%
6,poem-host pairs in model window,"252,253"
7,poem-host pairs represented by modelled poems,"93,358"
8,share of poem-host pairs represented,37.0%
9,host works touched by modelled poems,"3,619"


### 1.3 Poem length... does it matter?

Passim finds excerpts by matching text passages. A poem with 10K lines has far more textual surface area that *could* match against a host work's text than a 14-line sonnet. This creates a structural **length bias**: longer poems may appear in more host works simply because there are more opportunities for a passage to match, not because they were more fashionable. That bias shifts the poem-level intercept (baseline prevalence) in proportion to $\log(\text{length})$, but — crucially — it does *not* shift the slope over time, because length is time-invariant per poem. So our risers-and-decliners story in notebook 04 is largely insulated from length; the ranking of poems by *lifetime* volume is not.

The diagnostic below checks how strong the length-versus-volume relationship actually is. If it is tight, length is doing a lot of the work behind "canonical" poems appearing at the top of lifetime-works lists. If it is loose, the literary-historical signal dominates. Either way we pass the result forward into notebook 04 as context.

In [5]:
_poem_meta_path = data_paths["poem_meta"]
_pm = pl.read_csv(str(_poem_meta_path))
_pm_sub = _pm.select(
    pl.col("poem_id"),
    pl.col("num_lines").cast(pl.Int64, strict=False).alias("num_lines"),
    pl.col("num_words").cast(pl.Int64, strict=False).alias("num_words"),
    pl.col("char_len").cast(pl.Int64, strict=False).alias("char_len"),
).unique(subset=["poem_id"])

_length_df = _modeled_poems_pl.to_pandas().merge(
    _pm_sub.to_pandas(), on="poem_id", how="left"
)
_length_df["log_lines"] = np.log1p(_length_df["num_lines"].fillna(0))
_length_df["log_words"] = np.log1p(_length_df["num_words"].fillna(0))
_has_length = _length_df["num_lines"].notna() & (_length_df["num_lines"] > 0)
_corr_lines = _length_df.loc[_has_length, ["log_lines", "lifetime_works"]].corr().iloc[0, 1]
_corr_words = _length_df.loc[_has_length, ["log_words", "lifetime_works"]].corr().iloc[0, 1]
print(f"Length-bias diagnostic (support-qualified poems with length data: {_has_length.sum()}/{len(_length_df)}):")
print(f"  Pearson r(log lines, lifetime works)  = {_corr_lines:.3f}")
print(f"  Pearson r(log words, lifetime works)  = {_corr_words:.3f}")
if abs(_corr_lines) > 0.5:
    print("  Moderate-to-strong length")
elif abs(_corr_lines) > 0.3:
    print("  Mild length bias")
else:
    print("  Length bias is weak")

_lb_scatter = (
    alt.Chart(_length_df[_has_length])
    .mark_circle(opacity=0.45, size=40, color="#4c78a8")
    .encode(
        x=alt.X("log_lines:Q", title="log(number of lines)"),
        y=alt.Y("lifetime_works:Q", title="Lifetime quoted works"),
        tooltip=[
            "poem_label",
            alt.Tooltip("num_lines:Q", title="lines"),
            alt.Tooltip("num_words:Q", title="words"),
            "lifetime_works:Q",
        ],
    )
    .properties(
        width=400,
        height=260,
        title=alt.TitleParams(
            f"Poem length vs prevalence (r = {_corr_lines:.3f})",
            subtitle="Each dot = one support-qualified poem | log scale on x-axis",
        ),
    )
)
_lb_scatter

Length-bias diagnostic (support-qualified poems with length data: 906/906):
  Pearson r(log lines, lifetime works)  = 0.303
  Pearson r(log words, lifetime works)  = 0.307
  Mild length bias


alt.Chart(...)

## 2. The primary model — Bayesian hierarchical beta-binomial GLMM

*For each poem in each decade, what is the probability that a given PPA host work published in that decade quotes the poem at least once?* That is the quantity we model, and it is the cleanest statistical proxy for "is this poem in fashion?" that our evidence can support. Everything in notebook 04 — every risers-and-decliners table, every small-multiples panel, every author and period-level posterior — reads from this fit.

### 2.1 Specification and priors

The primary model is a **Bayesian hierarchical beta-binomial generalised linear mixed model**, fitted with [Bambi](https://bambinos.github.io/bambi/) on top of PyMC. In Bambi formula syntax:

```
p(quoted_works, n_works) ~ bs(period_c, df=4)
                         + (1 + period_z | poem_id)
                         + (1 | author)
family: beta_binomial
```

On the logit scale, for poem $i$ by author $a(i)$ in period $t$:

$$
\text{logit}\big(\mathbb{E}[p_{i,t}]\big) \;=\;
\underbrace{\alpha \;+\; \text{bs}(\text{period}_c, 4)_t}_{\text{shared time curve}}
\;+\; \underbrace{b^0_i \;+\; b^1_i \cdot \text{period}_z}_{\text{poem-level}}
\;+\; \underbrace{u_{a(i)}}_{\text{author-level}}
$$

with $\big(\text{quoted\_works}_{i,t} \,\big|\, p_{i,t}\big) \sim \text{BetaBinomial}(n_\text{works,t},\, p_{i,t},\, \kappa)$. Term by term:

- **`bs(period_c, df=4)`** is a B-spline on centred period with four degrees of freedom. It is the **shared** time curve — the "how is the whole PPA quoting poetry in period *t*?" signal that every poem inherits. Every small-multiples panel in notebook 04 shows its trace as the grey baseline dashed line.
- **`(1 + period_z | poem_id)`** gives each poem two random effects: an intercept $b^0_i$ (the poem's baseline prevalence — how much *above or below* the corpus average it sits, on average) and a slope $b^1_i$ (the poem's log-odds change per decade on top of the shared curve). The two are drawn jointly, so Bayesian shrinkage can pool information across poems. Sparse poems with noisy unregularised slopes are pulled toward the population mean; poems with rich data speak for themselves. 
- (I also tested a per-poem quadratic term `I(period_z**2) | poem_id`; its estimated SD was ~0.007 log-odds — two orders of magnitude smaller than the slope SD, so statistically irrelevant, and it roughly doubled sampling time. Dropping it changes no downstream conclusion.)
- **`(1 | author)`** is a random intercept per author, capturing correlated quoting of poems by the same author (editors bundle Pope with Dryden; some poets are systematically cited alongside their peers).
- **`beta_binomial`** absorbs extra-binomial variance (anthology clustering, genre fashions, measurement noise) inside the likelihood rather than inflating standard errors post-hoc. The concentration parameter $\kappa$ is learned from the data: large $\kappa$ means the beta-binomial is near-identical to a plain binomial; small $\kappa$ means there is substantial extra dispersion that a binomial could not accommodate.

All regression coefficients get weakly-informative Normal priors, all random-effect standard deviations get HalfNormal priors, and $\kappa$ gets a HalfCauchy(1) prior. Bambi sets the widths automatically to match the data scale.

### 2.2 Sampling and convergence

The modelling knobs below control how the sampler runs. We use NUTS (No-U-Turn Sampler) with 4 chains, 2000 tuning iterations, 1000 draws per chain, and `target_accept=0.95`. Typical runtime is 5–10 minutes on a laptop (depending of course on the params set earlier). After sampling we report R-hat and ESS for every parameter; anything with R-hat above 1.01 or ESS_bulk below 400 flags a convergence concern.

`RUN_BAMBI` is a top-level toggle — flip to `False` if you just want the rest of the notebook to run on cached results (or on a previous fit). `BAMBI_DRAWS / BAMBI_TUNE / BAMBI_CHAINS / BAMBI_TARGET_ACCEPT` mirror PyMC's standard NUTS arguments.

In [6]:
import time as _time

# toggle to actually run - otherwise we skip it
RUN_BAMBI = True

BAMBI_DRAWS = 2000 # quick run: 500, default 1000
BAMBI_TUNE = 3000 # quick run: 1000, default 2000, even better: 3000
BAMBI_CHAINS = 4
BAMBI_TARGET_ACCEPT = 0.95

if not RUN_BAMBI:
    print("Bambi model is DISABLED (RUN_BAMBI = False).")
    print(f"To enable, set RUN_BAMBI = True above. "
          f"Expected runtime: ~{BAMBI_DRAWS * BAMBI_CHAINS / 20:.0f}–{BAMBI_DRAWS * BAMBI_CHAINS / 6:.0f} minutes on a laptop.")
else:
    import os as _os
    _os.environ.setdefault(
        "MPLCONFIGDIR", str(CACHE_DIR / "mpl")
    )
    _os.environ.setdefault(
        "PYTENSOR_FLAGS",
        f"base_compiledir={CACHE_DIR}/pytensor",
    )
    import bambi as bmb
    import arviz as az

    print(f"Using bambi {bmb.__version__}, arviz {az.__version__}")

    _bambi_pm_path = data_paths["poem_meta"]
    _bambi_authors = (
        pl.read_csv(str(_bambi_pm_path))
        .select(
            pl.col("poem_id"),
            pl.col("author").cast(pl.Utf8, strict=False).fill_null("Unknown").str.strip_chars().alias("author"),
        )
        .unique(subset=["poem_id"])
        .to_pandas()
    )

    _bambi_df = _prev_df.merge(_bambi_authors, on="poem_id", how="left")
    _bambi_df["author"] = _bambi_df["author"].fillna("Unknown")
    _bambi_df["quoted_works"] = _bambi_df["quoted_works"].astype(int)
    _bambi_df["n_works"] = _bambi_df["n_works"].astype(int)
    _bambi_df = _bambi_df[_bambi_df["n_works"] > 0].reset_index(drop=True)

    print(
        f"Bambi dataset: {len(_bambi_df):,} rows | "
        f"{_bambi_df['poem_id'].nunique()} poems | "
        f"{_bambi_df['author'].nunique()} authors"
    )
    print(
        f"Settings: family=beta_binomial | chains={BAMBI_CHAINS} | "
        f"tune={BAMBI_TUNE} | draws={BAMBI_DRAWS} | target_accept={BAMBI_TARGET_ACCEPT}"
    )

    _bambi_formula = (
        "p(quoted_works, n_works) ~ "
        "bs(period_c, df=4) "
        "+ (1 + period_z | poem_id) "
        "+ (1 | author)"
    )
    print(f"\nFormula: {_bambi_formula}")

    print("\nBuilding model …")
    _t_build = _time.time()
    _bambi_model = bmb.Model(
        _bambi_formula,
        data=_bambi_df,
        family="beta_binomial",
    )
    print(f"  built in {_time.time() - _t_build:.1f}s")
    print("Model description:")
    print(_bambi_model)

    print("\nSampling — a PyMC/NUTS progress bar will appear below:")
    _t_fit = _time.time()
    _bambi_idata = _bambi_model.fit(
        draws=BAMBI_DRAWS,
        tune=BAMBI_TUNE,
        chains=BAMBI_CHAINS,
        cores=1, # delete this line to use all cores; you need a lot of RAM though!
        target_accept=BAMBI_TARGET_ACCEPT,
        progressbar=True,
        random_seed=42,
        idata_kwargs={"log_likelihood": False},
    )
    _fit_secs = _time.time() - _t_fit
    print(f"\nSampling finished in {_fit_secs/60:.1f} minutes ({_fit_secs:.1f}s).")

    print("\nConvergence diagnostics (R-hat and ESS):")
    _summ = az.summary(_bambi_idata, kind="diagnostics")
    _bad_rhat = _summ[_summ["r_hat"] > 1.01]
    _bad_ess = _summ[_summ["ess_bulk"] < 400]
    print(f"  n parameters: {len(_summ)}")
    print(f"  R-hat > 1.01: {len(_bad_rhat)} parameters")
    print(f"  ESS_bulk < 400: {len(_bad_ess)} parameters")
    print(f"  max R-hat: {_summ['r_hat'].max():.3f}")
    print(f"  min ESS_bulk: {_summ['ess_bulk'].min():.0f}")
    if len(_bad_rhat) == 0 and len(_bad_ess) == 0:
        print("  ✓ Sampling converged cleanly.")
    else:
        print("  ⚠ Some parameters did not fully converge; increase tune/draws or target_accept.")

Using bambi 0.17.2, arviz 0.23.1
Bambi dataset: 17,278 rows | 906 poems | 333 authors
Settings: family=beta_binomial | chains=4 | tune=3000 | draws=2000 | target_accept=0.95

Formula: p(quoted_works, n_works) ~ bs(period_c, df=4) + (1 + period_z | poem_id) + (1 | author)

Building model …
  built in 0.1s
Model description:
       Formula: p(quoted_works, n_works) ~ bs(period_c, df=4) + (1 + period_z | poem_id) + (1 | author)
        Family: beta_binomial
          Link: mu = logit
  Observations: 17278
        Priors: 
    target = mu
        Common-level effects
            Intercept ~ Normal(mu: 0.0, sigma: 6.1358)
            bs(period_c, df=4) ~ Normal(mu: [0. 0. 0. 0.], sigma: [12.5806 14.0858 10.6508  9.1092])
        
        Group-level effects
            1|poem_id ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 6.1358))
            period_z|poem_id ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 0.3904))
            1|author ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 6.1358))
        


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [kappa, Intercept, bs(period_c, df=4), 1|poem_id_sigma, 1|poem_id_offset, period_z|poem_id_sigma, period_z|poem_id_offset, 1|author_sigma, 1|author_offset]


Output()

Sampling 4 chains for 3_000 tune and 2_000 draw iterations (12_000 + 8_000 draws total) took 7670 seconds.



Sampling finished in 128.0 minutes (7680.7s).

Convergence diagnostics (R-hat and ESS):
  n parameters: 2154
  R-hat > 1.01: 0 parameters
  ESS_bulk < 400: 0 parameters
  max R-hat: 1.010
  min ESS_bulk: 621
  ✓ Sampling converged cleanly.


## 3. Does the model hold up?

A model that converges cleanly and reports small standard errors can still be wrong. The next cells run a few quality checks:

- is the overdispersion that motivated the beta-binomial real (we fit a quasi-binomial companion)?
- are the per-poem slopes the Bayesian model produces consistent with what an unpooled frequentist method would say?
- and does the model generate data that resemble what we actually observe?

These produce *only* diagnostics; all the primary visualisations live in notebook 04 and read from the Bayesian fit. (The original notebook also fit a conditional-intensity / *depth* model here; it has been dropped, since the paper argues about breadth only.)

### 3.1 Quasi-binomial companion

As a first sanity check we fit a **quasi-binomial fixed-effects GLM** on the same left-truncated poem × decade grid, with poem fixed effects and per-poem cubic-spline deviations. This is an unpooled frequentist model, meaning: every poem's intercept and slope are estimated from its own data, with no partial pooling across the corpus. On the logit scale:

$$
\text{logit}\big(p_{i,t}\big) \;=\; \text{cr}(\text{period}_c, \text{df})_t \;+\; \beta_i^{(\text{poem})} \;+\; \gamma_i(t)
$$

where $\beta_i^{(\text{poem})}$ is a per-poem fixed effect and $\gamma_i(t)$ is a per-poem deviation from the shared trend (`POEM_DEVIATION_TYPE` controls the basis — cubic spline, quadratic, or linear). We fit by IRLS and extract the quasi-binomial scale $\varphi = \chi^2 / df_{\text{resid}}$. 

But why fit this model when we already have one?

1. A $\varphi$ well above 1 (typically around 1.4 here) is exactly the evidence that motivates the beta-binomial family in §2 — the binomial alone cannot accommodate this much extra variance. The beta-binomial absorbs it natively; the quasi-binomial absorbs it only post-hoc by inflating standard errors by $\sqrt{\varphi}$.
2. Quasi-binom produces per-poem slopes with zero shrinkage. Comparing them to the Bayesian posterior means in §3.3 is the cleanest way to see how much partial pooling the hierarchical model is doing.
3. It's simply a nice non-Bayesian fallback that doesn't depend on NUTS convergence; so if §2 has sampled suspiciously, this cell is a deterministic second opinion.

This code and subsection produces *only* diagnostics and fitted values; all primary visualisations live in notebook 04 and use the Bayesian model.

In [7]:
from scipy.special import expit as _expit
import time as _time

def _build_poem_deviation_term(dev_type: str, dev_df: int) -> str:
    if dev_type == "linear":
        return "C(poem_id):period_z"
    if dev_type == "quadratic":
        return "C(poem_id):period_z + C(poem_id):I(period_z**2)"
    if dev_type == "cubic_spline":
        return f"C(poem_id):cr(period_z, df={dev_df})"
    raise ValueError(dev_type)

_dev_term = _build_poem_deviation_term(POEM_DEVIATION_TYPE, POEM_DEVIATION_DF)

print("Prevalence — full-model spline df search (quasi-binomial, scale='X2'):")
print(f"  Poem deviation: {POEM_DEVIATION_TYPE}"
      f"{' df=' + str(POEM_DEVIATION_DF) if POEM_DEVIATION_TYPE == 'cubic_spline' else ''}")
print("  Each candidate fits the complete model with poem FEs + poem-specific deviation.")
print("  Selection criterion: estimated dispersion φ closest to 1.0\n")

_prev_search_rows = []
_prev_search_fits = {}
for _sdf in PREV_SPLINE_DF_GRID:
    _t0 = _time.time()
    _formula_try = (
        f"cr(period_c, df={_sdf}) + C(poem_id) + {_dev_term}"
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _X_try = patsy.dmatrix(_formula_try, _prev_df, return_type="dataframe")
        _fit_try = sm.GLM(
            _prev_df["prop_quoted_raw"],
            _X_try,
            family=sm.families.Binomial(),
            var_weights=_prev_df["n_works"],
        ).fit(scale="X2", maxiter=100)
    _elapsed = _time.time() - _t0
    _prev_search_rows.append(
        {
            "df": _sdf,
            "scale_hat": _fit_try.scale,
            "pearson_disp": _fit_try.pearson_chi2 / _fit_try.df_resid,
            "n_params": _X_try.shape[1],
            "secs": round(_elapsed, 1),
        }
    )
    _prev_search_fits[_sdf] = (_fit_try, _X_try)
    print(f"  df={_sdf}: φ={_fit_try.scale:.3f}, {_X_try.shape[1]} params, {_elapsed:.1f}s")

_prev_spline_table = pd.DataFrame(_prev_search_rows).sort_values("df").reset_index(drop=True)
_prev_spline_table["dist_from_1"] = (_prev_spline_table["scale_hat"] - 1.0).abs()
PREV_SPLINE_DF = int(
    _prev_spline_table.sort_values("dist_from_1").iloc[0]["df"]
)
print(f"\n  Selected shared spline df = {PREV_SPLINE_DF} (φ closest to 1.0)")
print(
    _prev_spline_table[["df", "scale_hat", "pearson_disp", "n_params", "secs"]]
    .to_string(index=False, float_format=lambda v: f"{v:.3f}")
)

_prev_glm, _prev_X = _prev_search_fits[PREV_SPLINE_DF]
_prev_formula = f"cr(period_c, df={PREV_SPLINE_DF}) + C(poem_id) + {_dev_term}"
print(f"\nFinal prevalence formula: {_prev_formula}")

_prev_pred = _prev_glm.get_prediction(_prev_X).summary_frame(alpha=0.05)
_prev_model_df = _prev_df.copy()
_prev_model_df["p_fit"] = np.clip(_prev_pred["mean"].values, _PLOT_FLOOR, 1 - _PLOT_FLOOR)
_prev_model_df["p_lo"] = np.clip(_prev_pred["mean_ci_lower"].values, _PLOT_FLOOR, 1 - _PLOT_FLOOR)
_prev_model_df["p_hi"] = np.clip(_prev_pred["mean_ci_upper"].values, _PLOT_FLOOR, 1 - _PLOT_FLOOR)
_prev_model_df["pearson_r"] = _prev_glm.resid_pearson

_baseline_all = (
    _prev_model_df.groupby("period", as_index=False)
    .agg(
        baseline_fit_all=("p_fit", "mean"),
        baseline_raw_all=("prop_quoted_raw", "mean"),
        n_poems_active=("poem_id", "nunique"),
    )
    .sort_values("period")
    .reset_index(drop=True)
)

_cohort_obs_count = _prev_model_df.groupby("poem_id")["period"].nunique()
_max_periods = int(_prev_model_df["period"].nunique())
_cohort_pids = set(_cohort_obs_count[_cohort_obs_count >= _max_periods - 1].index)

_cohort_df = _prev_model_df[_prev_model_df["poem_id"].isin(_cohort_pids)].copy()
_baseline_cohort = (
    _cohort_df.groupby("period", as_index=False)
    .agg(
        baseline_fit_cohort=("p_fit", "mean"),
    )
    .sort_values("period")
    .reset_index(drop=True)
)

_prev_baseline = _baseline_all.merge(_baseline_cohort, on="period", how="left")
_prev_baseline["baseline_fit"] = _prev_baseline["baseline_fit_cohort"].fillna(
    _prev_baseline["baseline_fit_all"]
)
_prev_baseline["baseline_raw"] = _prev_baseline["baseline_raw_all"]

print(
    f"  stable cohort (present ≥{_max_periods - 1} periods): "
    f"{len(_cohort_pids)} poems out of {_prev_model_df['poem_id'].nunique()}"
)

_prev_model_df = _prev_model_df.merge(
    _prev_baseline[["period", "baseline_fit"]], on="period", how="left"
)
_prev_model_df["dev_value"], _prev_dev_title, _prev_dev_ref = _prevalence_dev_values(
    _prev_model_df["p_fit"],
    _prev_model_df["baseline_fit"],
    PREVALENCE_DEVIATION_SCALE,
)
_prev_model_df["risk_ratio_vs_baseline"] = (
    _prev_model_df["p_fit"] / _prev_model_df["baseline_fit"].clip(lower=_PLOT_FLOOR)
)

_prev_top_resid = _prev_model_df.reindex(
    _prev_model_df["pearson_r"].abs().sort_values(ascending=False).index
).head(8)
print("\nQuasi-binomial prevalence model fit:")
print(f"  estimated scale (φ) = {_prev_glm.scale:.3f}  (1.0 = no overdispersion)")
print(f"  SE inflation factor = √φ = {np.sqrt(_prev_glm.scale):.3f}")
print(f"  Pearson disp        = {_prev_glm.pearson_chi2 / _prev_glm.df_resid:.3f}")
print(f"  n params            = {_prev_X.shape[1]}")
print(
    f"  baseline (average across modeled poems) range = "
    f"{_prev_baseline['baseline_fit'].min():.5f} – {_prev_baseline['baseline_fit'].max():.5f}"
)
print(
    f"  baseline poems-active range = "
    f"{int(_prev_baseline['n_poems_active'].min())} – "
    f"{int(_prev_baseline['n_poems_active'].max())} poems per decade"
)
print("\nLargest |Pearson residual| poem × period cells (prevalence):")
print(
    _prev_top_resid[
        ["poem_label", "period", "quoted_works", "n_works", "prop_quoted_raw", "pearson_r"]
    ].to_string(index=False)
)


Prevalence — full-model spline df search (quasi-binomial, scale='X2'):
  Poem deviation: cubic_spline df=3
  Each candidate fits the complete model with poem FEs + poem-specific deviation.
  Selection criterion: estimated dispersion φ closest to 1.0

  df=5: φ=1.313, 3629 params, 1416.4s

  Selected shared spline df = 5 (φ closest to 1.0)
 df  scale_hat  pearson_disp  n_params     secs
  5      1.313         1.314      3629 1416.400

Final prevalence formula: cr(period_c, df=5) + C(poem_id) + C(poem_id):cr(period_z, df=3)
  stable cohort (present ≥23 periods): 403 poems out of 906

Quasi-binomial prevalence model fit:
  estimated scale (φ) = 1.313  (1.0 = no overdispersion)
  SE inflation factor = √φ = 1.146
  Pearson disp        = 1.314
  n params            = 3629
  baseline (average across modeled poems) range = 0.01555 – 0.03745
  baseline poems-active range = 339 – 906 poems per decade

Largest |Pearson residual| poem × period cells (prevalence):
                                  

### 3.2 Unpooled slopes with bootstrap CIs

From the quasi-binomial fit we extract a single scalar per poem: the OLS slope of its fitted logit trajectory against `period_z`. This is a compressed summary of each poem's unpooled trend, in log-odds per decade. To attach uncertainty we use a **parametric bootstrap**: 200 draws from the multivariate-normal approximation of the fitted coefficient vector, re-deriving each poem's slope on every draw. The 2.5th and 97.5th percentiles of those 200 slopes give a 95 % confidence interval. A slope whose CI excludes zero is one the unpooled model thinks is distinguishable from a flat trajectory, after the quasi-binomial's overdispersion correction.

The text tables below list the top 12 risers and decliners **under the sensitivity model**. The same rankings are re-computed from the Bayesian model in notebook 04, and the two sets are compared directly in the shrinkage diagnostic (§3.3). A slope of, say, +0.25 log-odds per decade translates to an odds ratio of $e^{0.25} \approx 1.28$ — the poem's odds of being quoted go up by roughly 28 % per decade relative to the shared corpus trend.

In [8]:
# "Slope" here is deviation-type-agnostic: for each poem we fit an OLS regression
# of the fitted logit values against period_z within the poem's at-risk window.
# This works the same way whether the underlying deviation is linear, quadratic,
# or a cubic spline — the reported slope is the best linear summary of the poem's
# fitted trajectory in log-odds per decade.

if "_prev_glm" not in dir() or "_prev_X" not in dir():
    raise RuntimeError(
        "§3.1 (quasi-binomial companion) must run successfully before "
        "§3.2. If §3.1 crashed with a LinAlgError on the "
        "poem-specific spline, reduce POEM_DEVIATION_TYPE to 'linear' or "
        "'quadratic' in §1 and re-run both cells."
    )

_beta_hat = _prev_glm.params.values
_V_hat = _prev_glm.cov_params().values
_X_arr = _prev_X.values
_logit_hat = _X_arr @ _beta_hat

_slope_work_df = _prev_model_df[["poem_id", "poem_label", "lifetime_works", "period_z"]].copy()
_slope_work_df["logit_hat"] = _logit_hat

def _ols_slope(x, y):
    _x = np.asarray(x, dtype=float)
    _y = np.asarray(y, dtype=float)
    _xm = _x - _x.mean()
    _xvar = float(np.sum(_xm * _xm))
    if _xvar == 0 or len(_x) < 2:
        return 0.0
    return float(np.sum(_xm * (_y - _y.mean())) / _xvar)

_slope_records = []
_poem_row_idx_map = {}
for _pid, _g in _slope_work_df.groupby("poem_id"):
    _slope_hat = _ols_slope(_g["period_z"].values, _g["logit_hat"].values)
    _slope_records.append({
        "poem_id": _pid,
        "poem_label": _g["poem_label"].iloc[0],
        "lifetime_works": int(_g["lifetime_works"].iloc[0]),
        "slope_hat": _slope_hat,
        "n_periods_modeled": len(_g),
    })
    _poem_row_idx_map[_pid] = _g.index.to_numpy()

_slope_info = pd.DataFrame(_slope_records)

_N_BOOT = 200
np.random.seed(42)
print(f"Bootstrap: drawing {_N_BOOT} samples from MVN(β̂, V̂) and re-deriving each poem's OLS slope on the fitted logit…")
_t_boot = _time.time()
_boot_betas = np.random.multivariate_normal(_beta_hat, _V_hat, size=_N_BOOT)
_boot_logits = _boot_betas @ _X_arr.T

_slope_lo = np.zeros(len(_slope_info))
_slope_hi = np.zeros(len(_slope_info))
for _i, _row in _slope_info.iterrows():
    _rows_idx = np.array([_prev_model_df.index.get_loc(idx) for idx in _poem_row_idx_map[_row["poem_id"]]])
    _z = _prev_model_df["period_z"].values[_rows_idx]
    _zm = _z - _z.mean()
    _xvar = float(np.sum(_zm * _zm))
    if _xvar == 0:
        _slope_lo[_i] = 0.0
        _slope_hi[_i] = 0.0
        continue
    _logit_boot = _boot_logits[:, _rows_idx]
    _slopes_boot = np.sum(_zm[None, :] * (_logit_boot - _logit_boot.mean(axis=1, keepdims=True)), axis=1) / _xvar
    _slope_lo[_i] = np.percentile(_slopes_boot, 2.5)
    _slope_hi[_i] = np.percentile(_slopes_boot, 97.5)

_slope_info["slope_lo"] = _slope_lo
_slope_info["slope_hi"] = _slope_hi
_slope_info["significant"] = (_slope_info["slope_lo"] > 0) | (_slope_info["slope_hi"] < 0)
print(f"  bootstrap complete in {_time.time() - _t_boot:.1f}s")

# depth / conditional-intensity dropped in the refactor — breadth only

_slope_info = _slope_info.sort_values("slope_hat", ascending=False).reset_index(drop=True)

_disp_cols = [
    "poem_label", "slope_hat", "slope_lo", "slope_hi", "significant",
    "lifetime_works",
]
print(f"Parametric bootstrap: {_N_BOOT} draws from MVN(β̂, V̂) — quasi-binomial corrected")
print(f"Poems with significant (95% CI excludes 0) slope: "
      f"{_slope_info['significant'].sum()} / {len(_slope_info)}")
print("\nTop 12 risers (positive slope = log-odds per decade above shared trend):")
print(
    _slope_info.head(12)[_disp_cols]
    .assign(
        slope_hat=lambda d: d["slope_hat"].round(3),
        slope_lo=lambda d: d["slope_lo"].round(3),
        slope_hi=lambda d: d["slope_hi"].round(3),
    )
    .to_string(index=False)
)
print("\nTop 12 decliners (negative slope):")
print(
    _slope_info.tail(12).sort_values("slope_hat")[_disp_cols]
    .assign(
        slope_hat=lambda d: d["slope_hat"].round(3),
        slope_lo=lambda d: d["slope_lo"].round(3),
        slope_hi=lambda d: d["slope_hi"].round(3),
    )
    .to_string(index=False)
)


Bootstrap: drawing 200 samples from MVN(β̂, V̂) and re-deriving each poem's OLS slope on the fitted logit…
  bootstrap complete in 12.0s
Parametric bootstrap: 200 draws from MVN(β̂, V̂) — quasi-binomial corrected
Poems with significant (95% CI excludes 0) slope: 373 / 906

Top 12 risers (positive slope = log-odds per decade above shared trend):
                                                                              poem_label  slope_hat  slope_lo  slope_hi  significant  lifetime_works
                                                                               PSAL. XC.      1.069     0.009     1.836         True              34
                                                                          LAȜAMON'S BRUT      0.897     0.094     1.508         True              67
                                         A Description of a Summer-Night in the Country.      0.611     0.039     1.258         True              46
                                                         

### 3.3 Shrinkage — Bayesian vs quasi-binomial slopes

Partial pooling does two things at once. It lets the hierarchical model estimate poems with little data by borrowing strength from the rest of the corpus, and it regularises poems whose unpooled estimates are overconfident. The scatter below compares each poem's quasi-binomial slope (x-axis) with its Bayesian posterior-mean slope (y-axis). Points on the diagonal mean the two models agree; points pulled toward the horizontal zero line have been **shrunk** by the hierarchical model — typically poems with fewer observed decades whose QB estimates were noisy. Blue markers indicate poems whose Bayesian 95 % credible interval excludes zero. A tight diagonal with a small shrunk tail is what a well-specified hierarchical model should look like.

In [9]:
if not RUN_BAMBI or "_bambi_idata" not in dir():
    print("Bambi model not fitted yet — rerun §2 with RUN_BAMBI = True.")
else:
    _posterior = _bambi_idata.posterior

    _candidates = []
    for _v in _posterior.data_vars:
        if "poem_id" not in _v:
            continue
        if "I(" in _v or "**" in _v:
            continue
        if not _v.startswith("period_z") and "period_z|" not in _v:
            continue
        _arr = _posterior[_v]
        _extra_dims = [d for d in _arr.dims if d not in ("chain", "draw")]
        if len(_extra_dims) == 0:
            continue
        _candidates.append((_v, _extra_dims))

    if not _candidates:
        print("Available posterior data_vars:")
        for _v in _posterior.data_vars:
            print(f"  {_v}: dims={list(_posterior[_v].dims)}")
        raise RuntimeError("Could not locate per-poem period_z random slope in posterior.")

    _candidates.sort(key=lambda x: len(x[0]))
    _slope_var, _extra_dims = _candidates[0]
    _poem_dim = _extra_dims[0]
    print(f"Extracted poem-slope variable: {_slope_var}  (poem dim: {_poem_dim})")

    _slope_array = _posterior[_slope_var].stack(draws=("chain", "draw"))
    _pid_coord = list(_slope_array[_poem_dim].values)
    _slope_mean = _slope_array.mean(dim="draws").values
    _slope_sd = _slope_array.std(dim="draws").values
    _slope_vals = np.asarray(_slope_array.values)
    if _slope_vals.shape[0] != len(_pid_coord):
        _slope_vals = _slope_vals.T
    _slope_lo_bayes = np.percentile(_slope_vals, 2.5, axis=1)
    _slope_hi_bayes = np.percentile(_slope_vals, 97.5, axis=1)

    _bambi_slope_df = pd.DataFrame({
        "poem_id": _pid_coord,
        "slope_bayes": _slope_mean,
        "slope_bayes_sd": _slope_sd,
        "slope_bayes_lo": _slope_lo_bayes,
        "slope_bayes_hi": _slope_hi_bayes,
    })

    _compare_df = _slope_info.merge(
        _bambi_slope_df, on="poem_id", how="inner"
    )
    print(f"\nShrinkage summary over {len(_compare_df)} poems:")
    _ratio = _compare_df["slope_bayes"].abs() / _compare_df["slope_hat"].abs().clip(lower=1e-6)
    print(f"  median |Bayesian| / |quasi-binomial| = {_ratio.median():.3f}")
    print(f"  fraction with |Bayesian| < |quasi-binomial| (shrunk toward 0) = {(_ratio < 1).mean():.0%}")

    _compare_chart = (
        alt.Chart(_compare_df)
        .mark_circle(opacity=0.5)
        .encode(
            x=alt.X("slope_hat:Q", title="Quasi-binomial slope"),
            y=alt.Y("slope_bayes:Q", title="Bayesian posterior-mean slope (shrunk)"),
            size=alt.Size("lifetime_works:Q", title="Lifetime works",
                           scale=alt.Scale(range=[40, 300])),
            color=alt.condition(
                (alt.datum.slope_bayes_lo > 0) | (alt.datum.slope_bayes_hi < 0),
                alt.value("#4c78a8"),
                alt.value("#d3d3d3"),
            ),
            tooltip=[
                "poem_label",
                alt.Tooltip("slope_hat:Q", format=".3f"),
                alt.Tooltip("slope_bayes:Q", format=".3f"),
                alt.Tooltip("slope_bayes_lo:Q", format=".3f"),
                alt.Tooltip("slope_bayes_hi:Q", format=".3f"),
                "lifetime_works:Q",
            ],
        )
        .properties(
            width=500, height=500,
            title=alt.TitleParams(
                "Shrinkage: Bayesian slopes vs quasi-binomial slopes",
                subtitle="Off-diagonal = shrinkage | Blue = 95% credible interval excludes 0",
            ),
        )
    )

    _diag = (
        alt.Chart(pd.DataFrame({
            "x": [_compare_df["slope_hat"].min(), _compare_df["slope_hat"].max()],
            "y": [_compare_df["slope_hat"].min(), _compare_df["slope_hat"].max()],
        }))
        .mark_line(color="black", strokeDash=[4, 3], opacity=0.5, strokeWidth=1)
        .encode(x="x:Q", y="y:Q")
    )

    from IPython.display import display as _display
    _display(alt.layer(_diag, _compare_chart).resolve_scale(color="independent"))

Extracted poem-slope variable: period_z|poem_id  (poem dim: poem_id__factor_dim)

Shrinkage summary over 906 poems:
  median |Bayesian| / |quasi-binomial| = 0.757
  fraction with |Bayesian| < |quasi-binomial| (shrunk toward 0) = 60%


alt.LayerChart(...)

### 3.4 Variance partition and posterior predictive check

Two final checks.

**Variance partition.** The three random-effect standard deviations (`1 | poem_id`, `period_z | poem_id`, `1 | author`) tell us *where* poem-to-poem heterogeneity lives. A large poem-intercept SD means poems differ a lot in baseline prevalence; a large poem-slope SD means they differ a lot in direction of trend; a large author SD means authors cluster systematically. Relative sizes of these three numbers are the first thing to read when we ask "what drives the variation between poems?"

**Posterior predictive check.** We draw 200 fresh replications of `quoted_works` from the fitted model and compute three summary statistics on each draw (mean, variance, fraction of zeros). The 95 % predictive interval for each statistic should contain the observed value if the model is well specified. A miss on `frac_zero` with a hit on mean and variance typically indicates mild zero-inflation — some poem × period cells are structurally zero in a way the model has not captured.

In [10]:
if not RUN_BAMBI or "_bambi_idata" not in dir():
    print("Bambi model not fitted — rerun §2 with RUN_BAMBI = True.")
else:
    _sigma_vars = {
        "poem_id intercept": "1|poem_id_sigma",
        "poem_id slope":     "period_z|poem_id_sigma",
        "author intercept":  "1|author_sigma",
    }

    _sigma_rows = []
    for _label, _varname in _sigma_vars.items():
        if _varname not in _posterior.data_vars:
            continue
        _arr = np.asarray(_posterior[_varname].values).flatten()
        _sigma_rows.append({
            "component": _label,
            "var": _varname,
            "posterior_mean": float(_arr.mean()),
            "posterior_sd": float(_arr.std()),
            "ci_lo": float(np.percentile(_arr, 2.5)),
            "ci_hi": float(np.percentile(_arr, 97.5)),
        })
    _sigma_df = pd.DataFrame(_sigma_rows)
    print("Random-effect SDs (posterior summaries, log-odds scale):")
    print(
        _sigma_df.assign(
            posterior_mean=lambda d: d["posterior_mean"].round(4),
            posterior_sd=lambda d: d["posterior_sd"].round(4),
            ci_lo=lambda d: d["ci_lo"].round(4),
            ci_hi=lambda d: d["ci_hi"].round(4),
        )
        .to_string(index=False)
    )
    if "kappa" in _posterior.data_vars:
        _kappa_arr = np.asarray(_posterior["kappa"].values).flatten()
        print(
            f"\nkappa (beta-binomial concentration): mean = {_kappa_arr.mean():.3f}, "
            f"95% CI [{np.percentile(_kappa_arr, 2.5):.3f}, {np.percentile(_kappa_arr, 97.5):.3f}]"
        )
        print(
            f"  (higher kappa = closer to plain Binomial; lower kappa = more extra-binomial variance)"
        )

    _sigma_chart = (
        alt.Chart(_sigma_df)
        .mark_bar(color="#4c78a8", opacity=0.8)
        .encode(
            x=alt.X("component:N", title=None, sort=None, axis=alt.Axis(labelAngle=-20)),
            y=alt.Y("posterior_mean:Q", title="Posterior-mean SD (log-odds)"),
            tooltip=[
                "component", "var",
                alt.Tooltip("posterior_mean:Q", format=".3f"),
                alt.Tooltip("ci_lo:Q", format=".3f"),
                alt.Tooltip("ci_hi:Q", format=".3f"),
            ],
        )
        .properties(width=430, height=220, title="Variance partition — random-effect SDs")
    )
    _sigma_err = (
        alt.Chart(_sigma_df)
        .mark_rule(color="black", strokeWidth=1.5)
        .encode(
            x=alt.X("component:N", sort=None),
            y="ci_lo:Q",
            y2="ci_hi:Q",
        )
    )

    print("\nComputing posterior predictive draws (this may take ~20-40s)…")
    _t_ppc = _time.time()
    try:
        _bambi_model.predict(_bambi_idata, kind="response", inplace=True, random_seed=42)
        _ppc_done = True
    except Exception as _e_ppc:
        print(f"  PPC predict failed ({_e_ppc}); will skip PPC.")
        _ppc_done = False
    print(f"  posterior-predictive compute in {_time.time() - _t_ppc:.1f}s")

    _obs_q = _bambi_df["quoted_works"].values
    _obs_mean = float(np.mean(_obs_q))
    _obs_var = float(np.var(_obs_q))
    _obs_frac_zero = float(np.mean(_obs_q == 0))

    print("\nObserved quoted_works distribution:")
    print(f"  mean = {_obs_mean:.3f}, variance = {_obs_var:.3f}, frac zero = {_obs_frac_zero:.3f}")

    _ppc_stats_df = None
    if "posterior_predictive" in _bambi_idata.groups():
        _pp_group = _bambi_idata.posterior_predictive
        _pp_var = list(_pp_group.data_vars)[0]
        _pp = np.asarray(_pp_group[_pp_var].values).reshape(-1, len(_obs_q))
        _pp_means = _pp.mean(axis=1)
        _pp_vars = _pp.var(axis=1)
        _pp_zerofrac = (_pp == 0).mean(axis=1)
        print("Posterior-predictive summary (across draws):")
        print(
            f"  mean  — posterior mean = {_pp_means.mean():.3f}  95% PI "
            f"[{np.percentile(_pp_means, 2.5):.3f}, {np.percentile(_pp_means, 97.5):.3f}]"
        )
        print(
            f"  var   — posterior mean = {_pp_vars.mean():.3f}  95% PI "
            f"[{np.percentile(_pp_vars, 2.5):.3f}, {np.percentile(_pp_vars, 97.5):.3f}]"
        )
        print(
            f"  frac0 — posterior mean = {_pp_zerofrac.mean():.3f}  95% PI "
            f"[{np.percentile(_pp_zerofrac, 2.5):.3f}, {np.percentile(_pp_zerofrac, 97.5):.3f}]"
        )
        _ppc_stats_df = pd.DataFrame({
            "statistic": ["mean", "variance", "frac_zero"],
            "observed": [_obs_mean, _obs_var, _obs_frac_zero],
            "pp_mean": [_pp_means.mean(), _pp_vars.mean(), _pp_zerofrac.mean()],
            "pp_lo": [
                np.percentile(_pp_means, 2.5),
                np.percentile(_pp_vars, 2.5),
                np.percentile(_pp_zerofrac, 2.5),
            ],
            "pp_hi": [
                np.percentile(_pp_means, 97.5),
                np.percentile(_pp_vars, 97.5),
                np.percentile(_pp_zerofrac, 97.5),
            ],
        })
        _ppc_stats_df["in_ci"] = (
            (_ppc_stats_df["observed"] >= _ppc_stats_df["pp_lo"])
            & (_ppc_stats_df["observed"] <= _ppc_stats_df["pp_hi"])
        )
        print("\nPPC check: does observed statistic fall inside 95% predictive interval?")
        print(_ppc_stats_df.to_string(index=False))
    else:
        print("Posterior predictive samples not available — skipping PPC plot.")

    from IPython.display import display as _display

    _display(alt.layer(_sigma_chart, _sigma_err))

    if _ppc_stats_df is not None:
        _ppc_plot_df = pd.DataFrame({
            "quoted_works": np.asarray(_pp.flatten()),
            "source": "Posterior predictive",
        })
        _obs_plot_df = pd.DataFrame({
            "quoted_works": _obs_q,
            "source": "Observed",
        })
        _pp_plot_df = pd.concat([_ppc_plot_df.sample(min(20000, len(_ppc_plot_df)), random_state=0), _obs_plot_df])
        _ppc_hist = (
            alt.Chart(_pp_plot_df)
            .mark_bar(opacity=0.55)
            .encode(
                x=alt.X("quoted_works:Q", bin=alt.Bin(maxbins=40), title="quoted_works per poem × decade"),
                y=alt.Y("count():Q", title="Count"),
                color=alt.Color(
                    "source:N",
                    scale=alt.Scale(
                        domain=["Observed", "Posterior predictive"],
                        range=["#c45200", "#4c78a8"],
                    ),
                ),
            )
            .properties(
                width=540, height=220,
                title=alt.TitleParams(
                    "Posterior predictive check — observed vs simulated quoted_works",
                    subtitle="Overlap = well-specified likelihood | Systematic mismatch = model gap",
                ),
            )
        )
        _display(_ppc_hist)

Random-effect SDs (posterior summaries, log-odds scale):
        component                    var  posterior_mean  posterior_sd  ci_lo  ci_hi
poem_id intercept        1|poem_id_sigma          0.7296        0.0206 0.6901 0.7714
    poem_id slope period_z|poem_id_sigma          0.0976        0.0029 0.0922 0.1033
 author intercept         1|author_sigma          0.3449        0.0362 0.2774 0.4185

kappa (beta-binomial concentration): mean = 437.036, 95% CI [401.065, 475.835]
  (higher kappa = closer to plain Binomial; lower kappa = more extra-binomial variance)

Computing posterior predictive draws (this may take ~20-40s)…
  posterior-predictive compute in 128.0s

Observed quoted_works distribution:
  mean = 5.403, variance = 104.362, frac zero = 0.251
Posterior-predictive summary (across draws):
  mean  — posterior mean = 5.449  95% PI [5.386, 5.513]
  var   — posterior mean = 107.812  95% PI [103.480, 112.287]
  frac0 — posterior mean = 0.244  95% PI [0.238, 0.250]

PPC check: does obse

alt.LayerChart(...)

alt.Chart(...)

In [11]:
# --- Persist the fitted-model bundle for notebook 04 ------------------------
# Saves the posterior + the frames the findings notebook needs, so 04 never has
# to re-run the 5-10 min NUTS sample. Notebook 04 rebuilds the (unfitted) Bambi
# model from `bambi_df` and loads this idata to drive `.predict()`.
if RUN_BAMBI and "_bambi_idata" in dir():
    import json as _json, arviz as _az
    _MODEL_DIR = MODEL_DIR
    _MODEL_DIR.mkdir(parents=True, exist_ok=True)

    _az.to_netcdf(_bambi_idata, str(_MODEL_DIR / "bambi_idata.nc"))
    _bambi_df.to_parquet(_MODEL_DIR / "bambi_df.parquet")
    _prev_df.to_parquet(_MODEL_DIR / "prev_df.parquet")
    _prev_model_df.to_parquet(_MODEL_DIR / "prev_model_df.parquet")
    _compare_df.to_parquet(_MODEL_DIR / "compare_df.parquet")
    _slope_info.to_parquet(_MODEL_DIR / "slope_info.parquet")
    _modeled_poems_pl.write_parquet(_MODEL_DIR / "modeled_poems.parquet")
    with open(_MODEL_DIR / "model_meta.json", "w") as _f:
        _json.dump({
            "MODEL_YEAR_MIN": int(MODEL_YEAR_MIN),
            "POEM_PERIOD": POEM_PERIOD,
            "PREV_SPLINE_DF": int(PREV_SPLINE_DF),
            "bambi_formula": _bambi_formula,
            "PREVALENCE_DEVIATION_SCALE": PREVALENCE_DEVIATION_SCALE,
        }, _f, indent=1)
    print(f"wrote model bundle -> {_MODEL_DIR}")
else:
    print("RUN_BAMBI is False or model not fitted — nothing persisted.")

wrote model bundle -> /Users/wouter/gitrepos/quotable_canon/exports/model


The fitted beta-binomial posterior and its supporting frames are written to `exports/model/`. `04_findings.ipynb` loads the posterior back, rebuilds the model so it can predict, and turns the fit into the risers/decliners, archetype panels, and period views that carry the argument.